Làm sạch dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split

# --- 1. LOAD DỮ LIỆU ---
print("⏳ Đang đọc dữ liệu...")
try:
    df = pd.read_csv('mbti_1.csv')
    print(f"✅ Đã load xong! Tổng số dòng: {len(df)}")
except FileNotFoundError:
    print("❌ LỖI: Không tìm thấy file 'mbti_1.csv'. Chưa upload file lên mục Files bên trái!")

# --- 2. HÀM LÀM SẠCH (CLEANING) ---
def clean_text(text):
    # a. Chuyển chữ thường
    text = str(text).lower()

    # b. Xóa URL (Link)
    text = re.sub(r'http\S+', '', text)

    # c. Xóa dấu phân cách ||| của dataset này
    text = text.replace('|||', ' ')

    # d. 🚨 XÓA TỪ KHÓA MBTI (Chống lộ đề/Data Leakage)
    mbti_types = ['infj', 'entp', 'intp', 'intj', 'entj', 'enfj', 'infp', 'enfp',
                  'isfp', 'istp', 'isfj', 'istj', 'estp', 'esfp', 'estj', 'esfj']
    pattern = r'\b(?:' + '|'.join(mbti_types) + r')\b'
    text = re.sub(pattern, '', text)

    # e. Giữ lại chữ cái và khoảng trắng (Xóa số, emoji, dấu câu...)
    # Lưu ý: Baseline TF-IDF cần sạch nên xóa hết.
    text = re.sub(r'[^a-z\s]', ' ', text)

    # f. Xóa khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()

    return text

if 'df' in locals(): # Chỉ chạy nếu load được file
    print("⏳ Đang làm sạch văn bản...")
    df['clean_posts'] = df['posts'].apply(clean_text)
    print("✅ Làm sạch xong!")

    # --- 3. TẠO 4 CỘT NHÃN (LABEL ENCODING) ---
    print("⏳ Đang tách nhãn thành 4 trục...")
    def get_binary_labels(mbti_type):
        return pd.Series({
            'IE': 1 if 'I' in mbti_type else 0, # Introvert=1
            'NS': 1 if 'N' in mbti_type else 0, # Intuition=1
            'TF': 1 if 'T' in mbti_type else 0, # Thinking=1
            'JP': 1 if 'J' in mbti_type else 0  # Judging=1
        })

    binary_labels = df['type'].apply(get_binary_labels)
    df = pd.concat([df, binary_labels], axis=1)

    # --- 4. CHIA TRAIN/TEST VÀ LƯU FILE ---
    print("⏳ Đang chia file và lưu...")
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['type'])

    # Lưu ra file CSV
    train_df.to_csv('train_clean.csv', index=False)
    test_df.to_csv('test_clean.csv', index=False)

    print("\n🎉 Đã xong!")

⏳ Đang đọc dữ liệu...
✅ Đã load xong! Tổng số dòng: 8675
⏳ Đang làm sạch văn bản (Chờ khoảng 1-2 phút)...
✅ Làm sạch xong!
⏳ Đang tách nhãn thành 4 trục...
⏳ Đang chia file và lưu...

🎉 Đã xong!


Huấn luyện Baseline TF-IDF+ SVM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

# 1. KIỂM TRA DỮ LIỆU
# Nếu chưa có file clean thì load từ code làm sạch dữ liệu trên.
if 'train_df' not in locals():
    print("⏳ Đang load lại dữ liệu từ file csv...")
    try:
        train_df = pd.read_csv('train_clean.csv')
        test_df = pd.read_csv('test_clean.csv')
    except FileNotFoundError:
        print("❌ Lỗi: Không tìm thấy dữ liệu.")

# Xử lý trường hợp làm sạch quá kỹ ra chuỗi rỗng (NaN) -> thay bằng chuỗi trống
train_df['clean_posts'] = train_df['clean_posts'].fillna('')
test_df['clean_posts'] = test_df['clean_posts'].fillna('')

print(f"✅ Sẵn sàng train! Train size: {len(train_df)}, Test size: {len(test_df)}")

# 2. VECTOR HÓA (TF-IDF)
print("\n⏳ Đang Vector hóa (TF-IDF)...")
vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),      # Lấy cụm 1, 2 và 3 từ
    max_features=100000,     # Giới hạn 100k đặc trưng ok nhất
    sublinear_tf=True        # Giảm ảnh hưởng của từ lặp lại quá nhiều
)

# Học từ tập Train
X_train_vec = vectorizer.fit_transform(train_df['clean_posts'])
# Áp dụng lên tập Test (Tuyệt đối không fit lại)
X_test_vec = vectorizer.transform(test_df['clean_posts'])

print("✅ Vector hóa xong! Bắt đầu train 4 model...")

# 3. HUẤN LUYỆN VÒNG LẶP (4 Models)
target_cols = ['IE', 'NS', 'TF', 'JP']
target_names = {
    'IE': 'Introvert (1) vs Extrovert (0)',
    'NS': 'Intuition (1) vs Sensing (0)',
    'TF': 'Thinking (1) vs Feeling (0)',
    'JP': 'Judging (1) vs Perceiving (0)'
}

print("\n" + "="*50)
print("🚀 KẾT QUẢ BASELINE (TF-IDF + LinearSVC)")
print("="*50)

for col in target_cols:
    print(f"\n🔹 Đang train cặp: {col} [{target_names[col]}] ...")

    # Lấy nhãn
    y_train = train_df[col]
    y_test = test_df[col]

    # Khởi tạo mô hình SVM
    # class_weight='balanced'  để xử lý dữ liệu lệch
    clf = LinearSVC(class_weight='balanced', random_state=42, max_iter=1000)

    # Train
    clf.fit(X_train_vec, y_train)

    # Predict
    y_pred = clf.predict(X_test_vec)

    # Đánh giá
    acc = accuracy_score(y_test, y_pred)
    print(f"👉 Accuracy: {acc:.2%}")

    # In báo cáo chi tiết
    print(classification_report(y_test, y_pred, digits=4))

print("\n✅ HOÀN TẤT!")

✅ Sẵn sàng train! Train size: 6940, Test size: 1735

⏳ Đang Vector hóa (Bước này tốn RAM tí nhé)...
✅ Vector hóa xong! Bắt đầu train 4 model...

🚀 KẾT QUẢ BASELINE (TF-IDF + LinearSVC)

🔹 Đang train cặp: IE [Introvert (1) vs Extrovert (0)] ...
👉 Accuracy: 81.67%
              precision    recall  f1-score   support

           0     0.6812    0.3890    0.4952       401
           1     0.8373    0.9453    0.8880      1334

    accuracy                         0.8167      1735
   macro avg     0.7593    0.6672    0.6916      1735
weighted avg     0.8012    0.8167    0.7972      1735


🔹 Đang train cặp: NS [Intuition (1) vs Sensing (0)] ...
👉 Accuracy: 87.32%
              precision    recall  f1-score   support

           0     0.6667    0.1667    0.2667       240
           1     0.8806    0.9866    0.9306      1495

    accuracy                         0.8732      1735
   macro avg     0.7736    0.5766    0.5986      1735
weighted avg     0.8510    0.8732    0.8388      1735


🔹 Đang